In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install sentence-transformers
!pip install pypdf
!pip install transformers
!pip install accelerate
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00


In [16]:
!pip install langchain-core

In [4]:
from huggingface_hub import login
login()  # paste your HuggingFace token here

In [20]:
from google.colab import files
uploaded = files.upload()

Saving angular.pdf to angular.pdf


In [21]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader

docs = []

for file_name in uploaded.keys():
    ext = os.path.splitext(file_name)[1].lower()

    if ext == ".txt":
        print(f"📄 TXT detected: {file_name}")
        loader = TextLoader(file_name, encoding="utf-8")

    elif ext == ".pdf":
        print(f"📕 PDF detected: {file_name}")
        loader = PyPDFLoader(file_name)

    else:
        print(f"❌ Skipping: {file_name}")
        continue

    docs.extend(loader.load())

print("Total Documents:", len(docs))

📕 PDF detected: angular.pdf
Total Documents: 54


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(docs)

print("Total Chunks:", len(chunks))
print("\n--- Sample Chunk ---")
print(chunks[0].page_content)

Total Chunks: 178

--- Sample Chunk ---
Angular Interview Questions
To view the live version of the
page, click here.
© Copyright by Interviewbit


In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda"}  # use "cpu" if no GPU
)

print("✅ Embedding model loaded")

/tmp/ipykernel_926/3264714385.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded


In [23]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

print("✅ FAISS vector store created")

✅ FAISS vector store created


In [24]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # return top 3 relevant chunks
)

print("✅ Retriever ready")

✅ Retriever ready


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "google/gemma-2-2b-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

print("✅ Gemma-2 loaded")

Loading tokenizer...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✅ Gemma-2 loaded


In [32]:
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,        # ✅ required when using temperature
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id  # ✅ stops padding warning too
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ LLM pipeline ready")

Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'max_new_tokens', 'pad_token_id', 'temperature', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM pipeline ready


In [33]:
from langchain_core.prompts import PromptTemplate

template = """
You are a helpful assistant. Use the context below to answer the question.
If you don't know the answer from the context, just say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

print("✅ Prompt template ready")

✅ Prompt template ready


In [34]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Prompt ──────────────────────────────────────────────────
template = """
You are a helpful assistant. Use the context below to answer the question.
If you don't know the answer from the context, just say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# ── Format retrieved docs into single string ─────────────────
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── Build RAG Chain ──────────────────────────────────────────
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain ready!")

✅ RAG Chain ready!


In [35]:
def ask(question):
    print(f"\n❓ Question: {question}")
    print("-" * 50)
    result = rag_chain.invoke(question)
    print(f"💡 Answer:\n{result}")

# Test it
ask("diffrence between react and angular?")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



❓ Question: diffrence between react and angular?
--------------------------------------------------
💡 Answer:

You are a helpful assistant. Use the context below to answer the question.
If you don't know the answer from the context, just say "I don't know".

Context:
React can only be
used in UI
development only.
Angular features a wide wide range
of tools, libraries, frameworks,
plugins, and so on that make
development faster and more fun.
In React we can use
third-party libraries
for any features.
Angular uses Typescript. React uses Javascript.
5.   List out diﬀerences between AngularJS and Angular?
Page 9 © Copyright by Interviewbit

declarative language and is much easier to use than JavaScript.
Long-term Google support - Google announced Long-term support for Angular.
This means that Google plans to stick with Angular and further scale up its
ecosystem.
4.   What are the advantages of Angular over React?
Angular vs React:
Page 8 © Copyright by Interviewbit

Structure
While develo